## 1. Instalación de librerías

Este bloque de código instala las librerías necesarias para conectarnos y trabajar con MongoDB en Python. `pymongo` es la librería principal para interactuar con MongoDB, y `dnspython` es un soporte adicional para resolver nombres de dominio, útil para conexiones a bases de datos en la nube como MongoDB Atlas.

In [ ]:
pip install pymongo dnspython

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 26.1 MB/s eta 0:00:00


## 2. Conexión a MongoDB Atlas

Aquí nos conectamos a la base de datos MongoDB. Se define una `uri` (Uniform Resource Identifier) que es como la dirección de tu base de datos en la nube. Luego, se crea un cliente (`pymongo.MongoClient`) usando esa dirección y se selecciona la base de datos (`banco_db`) y la colección (`clientes`) con la que vamos a trabajar. La colección es como una tabla donde guardamos nuestros datos.

In [ ]:
import os
import pymongo

uri = os.getenv("MONGO_URI")

client = pymongo.MongoClient(uri)
db = client["banco_db"]
coleccion = db["clientes"]

print("Conexión realizada con éxito")

Conexión realizada con éxito


## 3. Insertar un nuevo cliente (Documento)

Este código crea un nuevo 'cliente' (que en MongoDB se llama 'documento') y lo inserta en la colección `clientes`. Un documento es como un registro de información, en este caso, de una persona con su nombre, edad, tipo de cuenta, saldo y si está activo. `insert_one` añade un solo documento.

In [ ]:
nuevo_cliente = {
    "nombre": "Roberto Gómez",
    "edad": 30,
    "tipo_cuenta": "ahorro",
    "saldo": 5000,
    "activo": True
}
resultado = coleccion.insert_one(nuevo_cliente)
print("Cliente creado con éxito")

Cliente creado con éxito


## 4. Encontrar un cliente por nombre

Aquí buscamos un solo cliente en la colección `clientes` utilizando su nombre. `find_one` nos devuelve el primer documento que coincide con el criterio de búsqueda (en este caso, `nombre: Roberto Gómez`).

In [ ]:
cliente = coleccion.find_one({"nombre": "Roberto Gómez"})
print(cliente)

{'_id': ObjectId('6aad1a8456fcf0c851af31ec'), 'nombre': 'Roberto Gómez', 'edad': 30, 'tipo_cuenta': 'ahorro', 'saldo': 5000, 'activo': True}


## 5. Buscar clientes con saldo mayor a un valor

Este código busca todos los clientes que tienen un saldo mayor a 3000. `find` se usa para buscar múltiples documentos. `$gt` significa 'mayor que' (greater than). Luego, se recorre el resultado (`cursor`) para imprimir el nombre y el saldo de cada cliente encontrado.

In [ ]:
cursor = coleccion.find({"saldo": {"$gt":3000}})
for doc in cursor:
  print(doc["nombre"],doc["saldo"])

Carlos Ruiz 12800
Javier López 45000
Sofía Martínez 3500.75
Roberto Gómez 5000


## 6. Actualizar un cliente: cambiar el estado de activo

Aquí actualizamos la información de un cliente específico. Buscamos al cliente con `nombre: Javier López` y usamos `$set` para cambiar su campo `activo` a `False`. `update_one` actualiza un solo documento que cumpla con el criterio.

In [ ]:
coleccion.update_one({"nombre": "Javier López"},{"$set": {"activo":False}} )

UpdateResult({'n': 1, 'electionId': ObjectId('7fffffff0000000000000057'), 'opTime': {'ts': Timestamp(1789729413, 45), 't': 87}, 'nModified': 0, 'ok': 1.0, '$clusterTime': {'clusterTime': Timestamp(1789729413, 46), 'signature': {'hash': b"\x86x\xd1?\x01m(\xff8X'\xe5\x9eN\xfc\x86}rW\xff", 'keyId': 7634928564827062277}}, 'operationTime': Timestamp(1789729413, 45), 'updatedExisting': True}, acknowledged=True)

## 7. Verificar la actualización del cliente

Después de actualizar a Javier López, este código vuelve a buscar su documento para confirmar que el campo `activo` ha cambiado a `False`.

In [ ]:
cliente2 = coleccion.find_one({"nombre": "Javier López"})
print(cliente2)

{'_id': ObjectId('6aa870ee3ebfbd1c265329d8'), 'nombre': 'Javier López', 'edad': 50, 'tipo_cuenta': 'premium', 'saldo': 45000, 'activo': False}


## 8. Actualizar múltiples clientes: incrementar saldo

Este código actualiza a varios clientes a la vez. Busca a todos los clientes cuyo `tipo_cuenta` es 'corriente' y usa `$inc` para incrementar su `saldo` en 200. `update_many` aplica el cambio a todos los documentos que coincidan con el criterio.

In [ ]:
resultado = coleccion.update_many(
    {"tipo_cuenta": "corriente"},
    {"$inc":{"saldo": 200}}
)
print(resultado.modified_count, "clientes actualizados")

3 clientes actualizados


## 9. Eliminar un cliente por nombre

Aquí eliminamos un cliente de la colección. `delete_one` borra el primer documento que coincide con el nombre 'Roberto Gómez'.

In [ ]:
coleccion.delete_one({"nombre" : "Roberto Gómez"})

DeleteResult({'n': 1, 'electionId': ObjectId('7fffffff0000000000000057'), 'opTime': {'ts': Timestamp(1789729414, 20), 't': 87}, 'ok': 1.0, '$clusterTime': {'clusterTime': Timestamp(1789729414, 20), 'signature': {'hash': b'\xbfx}\x1d?\xebu\x9a)\xf8\x0f\x0ff$G\xf3\xd7L\xd5\xe0', 'keyId': 7634928564827062277}}, 'operationTime': Timestamp(1789729414, 20)}, acknowledged=True)

## 10. Proyectar campos específicos en los resultados

Este código busca todos los documentos, pero solo muestra campos específicos: `nombre` y `edad`. El `_id: 0` significa que no queremos mostrar el campo `_id` que MongoDB añade automáticamente a cada documento. Esto es útil para obtener solo la información que realmente necesitas.

In [ ]:
cursor = coleccion.find({}, {"nombre" : 1, "edad":1, "_id": 0})
for doc in cursor:
  print(doc)

{}
{'nombre': 'Laura Gómez', 'edad': 28}
{'nombre': 'Carlos Ruiz', 'edad': 42}
{'nombre': 'Marta Fernández', 'edad': 35}
{'nombre': 'Javier López', 'edad': 50}
{'nombre': 'Sofía Martínez', 'edad': 23}


## 11. Ordenar clientes por saldo (descendente)

Aquí buscamos todos los clientes y los ordenamos por su `saldo` de mayor a menor (`pymongo.DESCENDING`). Luego, imprimimos su nombre y saldo. `get` se usa para acceder a los campos de forma segura, en caso de que algún documento no tenga ese campo.

In [ ]:
cursor = coleccion.find().sort("saldo", pymongo.DESCENDING)
for doc in cursor:
  print(doc.get("nombre", "Nombre no disponible"), doc.get("saldo", "Saldo no disponible"))

Javier López 45000
Carlos Ruiz 12800
Sofía Martínez 3700.75
Laura Gómez 3050.5
Marta Fernández 1490.1
Nombre no disponible Saldo no disponible


## 12. Contar clientes que cumplen una condición

Este código cuenta cuántos clientes tienen una `edad` mayor a 30 años. `count_documents` es una manera rápida de obtener el número de documentos que coinciden con un criterio.

In [ ]:
total = coleccion.count_documents({"edad":{"$gt": 30}})
print("edad total", total)

edad total 3


## 13. Agregación: Contar clientes por tipo de cuenta

Este bloque introduce la 'agregación', que es una forma potente de procesar los datos en MongoDB. Aquí, usamos el `$group` para agrupar a los clientes por su `tipo_cuenta` y luego contamos cuántos clientes hay en cada grupo con `$sum: 1`. Esto nos dice cuántos clientes tienen una cuenta de 'ahorro', 'corriente', etc.

In [ ]:
db.clientes.aggregate([
  {
    "$group": {
      "_id": "$tipo_cuenta",
      "total_clientes": { "$sum": 1 }
    }
  }
]);

## 14. Agregación: Contar clientes por tipo de cuenta (Forma corregida)

Este es el mismo ejemplo que el anterior, pero usando la forma correcta de llamar al método `aggregate` en la colección. La `pipeline` es una lista de etapas de procesamiento que MongoDB ejecuta una tras otra. En este caso, solo hay una etapa de `$group`.

In [ ]:
pipeline = [{"$group": {"_id": "$tipo_cuenta", "total_clientes": {"$sum": 1}}}]
for doc in coleccion.aggregate(pipeline):
  print(doc)

{'_id': 'ahorro', 'total_clientes': 1}
{'_id': None, 'total_clientes': 1}
{'_id': 'corriente', 'total_clientes': 3}
{'_id': 'premium', 'total_clientes': 1}


## 15. Agregación: Calcular el saldo medio por tipo de cuenta

Similar al ejemplo anterior, pero en lugar de contar clientes, calculamos el `saldo_medio` (saldo promedio) para cada `tipo_cuenta` usando el operador `$avg` en el campo `saldo`. Esto nos da una idea del saldo promedio en cada tipo de cuenta.

In [ ]:
pipeline = [
    {"$group": {"_id": "$tipo_cuenta", "total_clientes": {"$sum": 1}}}
]
for doc in coleccion.aggregate(pipeline):
    print(doc)


{'_id': None, 'total_clientes': 1}
{'_id': 'corriente', 'total_clientes': 3}
{'_id': 'premium', 'total_clientes': 1}
{'_id': 'ahorro', 'total_clientes': 1}


In [ ]:
pipeline = [
    {"$group": {"_id": "$tipo_cuenta", "saldo_medio": {"$avg": "$saldo"}}}
]
for doc in coleccion.aggregate(pipeline):
    print(doc["_id"], doc["saldo_medio"])


corriente 2747.116666666667
premium 45000.0
ahorro 12800.0
None None


## 16. Agregación: Saldo medio de clientes activos por tipo de cuenta

Aquí combinamos dos etapas de agregación: primero, `$match` para filtrar solo a los clientes que están `activo: True`. Luego, `$group` para calcular el `saldo_medio` por `tipo_cuenta` solo para esos clientes activos. Esto es útil para analizar subconjuntos de tus datos.

In [ ]:
pipeline = [
    {"$match": {"activo": True}},
    {"$group": {"_id": "$tipo_cuenta", "saldo_medio": {"$avg": "$saldo"}}}
]
for doc in coleccion.aggregate(pipeline):
    print(doc)


{'_id': 'ahorro', 'saldo_medio': 12800.0}
{'_id': 'corriente', 'saldo_medio': 2747.116666666667}


## 17. Agregación: Saldo total de clientes activos

En este ejemplo, usamos `$match` para filtrar clientes `activo: True` y luego `$group` para calcular el `saldo_total` de todos ellos, sin agrupar por un campo específico (`_id: None`). Esto nos da la suma total de saldos de todos los clientes activos.

In [ ]:
pipeline = [
    {"$match": {"activo": True}},
    {"$group" : {"_id": None, "saldo_total":{"$sum":"$saldo"}}}
]
for doc in coleccion.aggregate(pipeline):
    print("Saldo total activos:", doc["saldo_total"])

Saldo total activos: 21041.35


## 18. Agregación: Edad máxima y mínima de todos los clientes

Este código utiliza la agregación para encontrar la `edad_max` y `edad_min` de todos los clientes en la colección. El `_id: None` indica que estamos calculando estos valores para toda la colección, no agrupando por ningún campo en particular.

In [ ]:
pipeline = [
    {"$group": {"_id":None,
     "edad_max":{"$max": "$edad"},
     "edad_min":{"$min": "$edad"}          }}
]
for doc in coleccion.aggregate(pipeline):
    print(doc)

{'_id': None, 'edad_max': 50, 'edad_min': 23}


## 19. Agregación: Saldo total de clientes activos por tipo de cuenta, ordenado

Este es un ejemplo más completo de agregación. Primero, `$match` para filtrar clientes activos. Luego, `$group` para calcular el `gran_total` de saldo por `tipo_cuenta`. Finalmente, `$sort` para ordenar los resultados de mayor a menor saldo total (`-1` para descendente). Esto muestra qué tipos de cuenta activa tienen los mayores saldos agregados.

In [ ]:
pipeline = [
    {"$match": {"activo": True}},
    {"$group": {"_id": "$tipo_cuenta", "gran_total": {"$sum": "$saldo"}}},
    {"$sort": {"gran_total": -1}}
]
for doc in coleccion.aggregate(pipeline):
    print(doc)

{'_id': 'ahorro', 'gran_total': 12800}
{'_id': 'corriente', 'gran_total': 8241.35}


## 20. Agregación: Proyectar un nuevo campo calculado

Aquí usamos la etapa `$project` para transformar los documentos. Crea un nuevo campo `saldo_dolares` que es el `saldo` original multiplicado por 3 (simulando una conversión a dólares). El `_id: 0` es para excluir el campo `_id` original.

In [ ]:
pipeline = [
    {
        "$project": {
            "_id": 0,
            "nombre": 1,
            "saldo_dolares": {"$multiply": ["$saldo", 3]}
        }
    }
]
for doc in coleccion.aggregate(pipeline):
    print(doc)

{'saldo_dolares': None}
{'nombre': 'Laura Gómez', 'saldo_dolares': 9151.5}
{'nombre': 'Carlos Ruiz', 'saldo_dolares': 38400}
{'nombre': 'Marta Fernández', 'saldo_dolares': 4470.299999999999}
{'nombre': 'Javier López', 'saldo_dolares': 135000}
{'nombre': 'Sofía Martínez', 'saldo_dolares': 11102.25}


## 21. Agregación: Contar clientes mayores de 30 años

Este código utiliza `$match` para filtrar a los clientes con una `edad` mayor a 30, y luego `$count` para simplemente contar cuántos documentos cumplen esa condición. Es una forma eficiente de obtener solo un conteo.

In [ ]:
pipeline = [
    {"$match": {"edad": {"$gt": 30}}},
    {"$count": "total_mayores_30"}
]
for doc in coleccion.aggregate(pipeline):
    print(doc)

{'total_mayores_30': 3}


## 22. Agregación: Edad promedio por estado de actividad

Este ejemplo agrupa (`$group`) a los clientes por su estado `activo` (True, False o None) y calcula la `edad_promedio` para cada grupo. Esto nos dice, por ejemplo, si los clientes activos son en promedio más jóvenes o mayores que los inactivos.

In [ ]:
pipeline = [
    {"$group": {"_id": "$activo", "edad_promedio": {"$avg": "$edad"}}}
]
for doc in coleccion.aggregate(pipeline):
    print("Activo:", doc["_id"], "| Edad media:", doc["edad_promedio"])

Activo: None | Edad media: None
Activo: False | Edad media: 50.0
Activo: True | Edad media: 32.0


## 23. Agregación: Saldo promedio por tipo de cuenta con filtro post-agregación

Aquí, primero agrupamos (`$group`) por `tipo_cuenta` para obtener el `saldo_promedio` de cada una. Luego, usamos `$match` para filtrar los resultados de esa agregación, mostrando solo los tipos de cuenta cuyo `saldo_promedio` es mayor a 3000. Es como filtrar los resultados de un cálculo previo.

In [ ]:
pipeline = [
    {"$group": {"_id": "$tipo_cuenta", "saldo_promedio": {"$avg": "$saldo"}}},
    {"$match": {"saldo_promedio": {"$gt": 3000}}}
]
for doc in coleccion.aggregate(pipeline):
    print(doc)

{'_id': 'premium', 'saldo_promedio': 45000.0}
{'_id': 'ahorro', 'saldo_promedio': 12800.0}


## 24. Creación de un índice simple

Los índices son como el índice de un libro: ayudan a MongoDB a encontrar los datos más rápido. Este código crea un índice en el campo `nombre`. Cuando busques por `nombre`, la base de datos lo hará de forma mucho más eficiente. El `1` indica un orden ascendente.

In [ ]:
coleccion.create_index([("nombre", 1)])
print("indice creado")

indice creado


## 25. Creación de un índice compuesto

Un índice compuesto es un índice en múltiples campos. Aquí se crea un índice sobre `corriente` y `saldo`. Si tus búsquedas a menudo involucran estos dos campos juntos, este índice acelerará esas consultas. Ten en cuenta que el campo `corriente` probablemente es un error tipográfico y debería ser `tipo_cuenta` si el objetivo era indexar la cuenta corriente.

In [ ]:
coleccion.create_index([("corriente", 1), ("saldo", 1)])

'corriente_1_saldo_1'

## 26. Creación de un índice compuesto con orden específico

Este es otro ejemplo de índice compuesto, pero especificando el orden para cada campo: `tipo_cuenta` de forma ascendente (`pymongo.ASCENDING`) y `saldo` de forma descendente (`pymongo.DESCENDING`). Esto es útil para consultas que también necesitan ordenar los resultados por estos campos.

In [ ]:
coleccion.create_index([
    ("tipo_cuenta", pymongo.ASCENDING),
    ("saldo", pymongo.DESCENDING)
])
print("Índice compuesto creado.")

Índice compuesto creado.


## 27. Listar todos los índices de la colección

Este código nos permite ver todos los índices que existen en la colección `clientes`. Muestra el `name` (nombre del índice) y las `key` (los campos sobre los que está creado el índice).

In [ ]:
for indice in coleccion.list_indexes():
    print(indice["name"], "->", indice["key"])

_id_ -> SON([('_id', 1)])
nombre_1 -> SON([('nombre', 1)])
corriente_1_saldo_1 -> SON([('corriente', 1), ('saldo', 1)])
tipo_cuenta_1_saldo_-1 -> SON([('tipo_cuenta', 1), ('saldo', -1)])


## 28. Eliminar un índice por su nombre

Si un índice ya no es necesario o quieres cambiarlo, puedes eliminarlo. Este código elimina el índice llamado `nombre_1` (el que creamos anteriormente en el campo `nombre`).

In [ ]:
coleccion.drop_index("nombre_1")

## 29. Insertar un documento con un objeto embebido

MongoDB permite almacenar datos anidados o 'embebidos' dentro de un documento. Aquí, insertamos un cliente donde la `direccion` no es solo un texto, sino un objeto con `calle` y `ciudad`. Esto es útil para mantener la información relacionada agrupada.

In [ ]:
resultado = coleccion.insert_one({
    "nombre": "Elena Torres",
    "edad": 31,
    "tipo_cuenta": "ahorro",
    "saldo": 4200,
    "activo": True,
    "direccion": { "calle": "Gran Vía 12", "ciudad": "Madrid" }
})
print("Cliente embebido insertado:", resultado.inserted_id)

Cliente embebido insertado: 6aad200556fcf0c851af31ed


## 30. Buscar documentos por campos dentro de objetos embebidos

Este código muestra cómo buscar documentos cuando la información que buscas está dentro de un objeto anidado. Para encontrar clientes que viven en 'Madrid', usamos la notación de 'punto' (`direccion.ciudad`) para acceder al campo dentro del objeto `direccion`.

In [ ]:
cursor = coleccion.find({"direccion.ciudad" : "Madrid"})
for doc in cursor:
  print(doc["nombre"], doc["direccion"])

Elena Torres {'calle': 'Gran Vía 12', 'ciudad': 'Madrid'}


## 31. Añadir un array (lista) a un documento

Los documentos de MongoDB también pueden contener listas (arrays). Aquí, actualizamos el cliente 'Elena Torres' para añadirle un nuevo campo `transacciones` que es una lista de números. `$set` se usa para establecer el valor de este campo.

In [ ]:
coleccion.update_one(
    {"nombre": "Elena Torres"},
    {"$set": {"transacciones": [100, -50, 200]}}
)
print("Array de transacciones añadido.")

Array de transacciones añadido.


## 32. Añadir un elemento a un array existente

Si ya tienes un array en un documento y quieres añadir más elementos sin sobrescribir lo que ya hay, usas el operador `$push`. Este código añade el número `500` al array `transacciones` de Elena Torres.

In [ ]:
coleccion.update_one(
    {"nombre": "Elena Torres"},
    {"$push": {"transacciones": 500}}
)

UpdateResult({'n': 1, 'electionId': ObjectId('7fffffff0000000000000057'), 'opTime': {'ts': Timestamp(1789731060, 39), 't': 87}, 'nModified': 1, 'ok': 1.0, '$clusterTime': {'clusterTime': Timestamp(1789731060, 39), 'signature': {'hash': b'\xd6\xae\xdf;\x04\xc3\xdfe\xad\x97u\xa1\xd1\xeamFh\xff\xb7\xe5', 'keyId': 7634928564827062277}}, 'operationTime': Timestamp(1789731060, 39), 'updatedExisting': True}, acknowledged=True)

## 33. Buscar documentos por valores dentro de un array

Este código busca clientes que tienen al menos una `transaccion` con un valor mayor a 300. MongoDB puede buscar dentro de los arrays para encontrar documentos que contienen elementos que cumplen una condición específica.

In [ ]:
cursor =coleccion.find({"transacciones": {"$gt": 300}})
for doc in cursor:
  print(doc["nombre"], doc["transacciones"])

Elena Torres [100, -50, 200, [500], 500]
